In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")

df_test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv") 

submission_ids = df_test["PassengerId"]

In [3]:
# Understanding the data 

print(df.sample(5))

     PassengerId HomePlanet CryoSleep     Cabin  Destination   Age    VIP  \
635      0663_01      Earth     False   F/140/P  TRAPPIST-1e  44.0  False   
928      0992_04     Europa     False       NaN  55 Cancri e  35.0   True   
6971     7402_02     Europa      True   C/272/S  55 Cancri e  45.0  False   
8325     8889_01        NaN     False  G/1439/P  55 Cancri e  17.0  False   
5314     5680_01      Earth     False   E/359/P  TRAPPIST-1e  18.0  False   

      RoomService  FoodCourt  ShoppingMall   Spa  VRDeck               Name  \
635          86.0      525.0          16.0  49.0    39.0    Jeroy Buckentry   
928        8030.0      300.0           0.0   0.0  1436.0    Naosa Herpumble   
6971          0.0        0.0           0.0   0.0     0.0  Mesarid Stioutody   
8325          0.0        0.0         615.0   0.0     0.0   Erman Russelloyd   
5314          0.0        0.0        1609.0   0.0     0.0  Ryanne Blanglison   

      Transported  
635          True  
928         False  
69

In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB
None


In [5]:
print(df.describe())

               Age   RoomService     FoodCourt  ShoppingMall           Spa  \
count  8514.000000   8512.000000   8510.000000   8485.000000   8510.000000   
mean     28.827930    224.687617    458.077203    173.729169    311.138778   
std      14.489021    666.717663   1611.489240    604.696458   1136.705535   
min       0.000000      0.000000      0.000000      0.000000      0.000000   
25%      19.000000      0.000000      0.000000      0.000000      0.000000   
50%      27.000000      0.000000      0.000000      0.000000      0.000000   
75%      38.000000     47.000000     76.000000     27.000000     59.000000   
max      79.000000  14327.000000  29813.000000  23492.000000  22408.000000   

             VRDeck  
count   8505.000000  
mean     304.854791  
std     1145.717189  
min        0.000000  
25%        0.000000  
50%        0.000000  
75%       46.000000  
max    24133.000000  


In [6]:
# Here i am removing string columns 

df = df.drop(columns = ["PassengerId","Cabin","Name"])

df_test = df_test.drop(columns = ["PassengerId","Cabin","Name"])

y = df.pop("Transported")

print(df)

     HomePlanet CryoSleep    Destination   Age    VIP  RoomService  FoodCourt  \
0        Europa     False    TRAPPIST-1e  39.0  False          0.0        0.0   
1         Earth     False    TRAPPIST-1e  24.0  False        109.0        9.0   
2        Europa     False    TRAPPIST-1e  58.0   True         43.0     3576.0   
3        Europa     False    TRAPPIST-1e  33.0  False          0.0     1283.0   
4         Earth     False    TRAPPIST-1e  16.0  False        303.0       70.0   
...         ...       ...            ...   ...    ...          ...        ...   
8688     Europa     False    55 Cancri e  41.0   True          0.0     6819.0   
8689      Earth      True  PSO J318.5-22  18.0  False          0.0        0.0   
8690      Earth     False    TRAPPIST-1e  26.0  False          0.0        0.0   
8691     Europa     False    55 Cancri e  32.0  False          0.0     1049.0   
8692     Europa     False    TRAPPIST-1e  44.0  False        126.0     4688.0   

      ShoppingMall     Spa 

In [7]:
# Seprating training and testing data 

X_train,X_test,y_train,y_test = train_test_split(df,y,random_state=42,test_size=0.2)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn import set_config
set_config(transform_output='pandas')
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [9]:
# Here sepreating Numerical and Categorical Columns

cat_cols = [col for col in df.columns if df[col].dtype=='object']
num_cols = [col for col in df.columns if df[col].dtype=='float64']

print(cat_cols)
print(num_cols)

['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


In [10]:
# Here I'm making data suitable for ml model to train properly

num_pipeline = Pipeline(steps=[
    ("p1",SimpleImputer(strategy= "median")),
    ("p2",StandardScaler())
])

cat_pipeline = Pipeline (steps=[
    ("p1",SimpleImputer(strategy = "most_frequent")),
    ("p2",OneHotEncoder(sparse_output = False , handle_unknown = "ignore"))
])


In [11]:
transform = ColumnTransformer(transformers=[
    ("nums",num_pipeline,num_cols),
    ("cat",cat_pipeline,cat_cols)
] ,remainder = "passthrough")

In [12]:
# a little bit of tuning of model after testing

model = RandomForestClassifier(random_state=42,n_estimators=400,
    max_depth=10)


In [13]:

full_pipeline = Pipeline(steps=[
    ("Transforming_data",transform),
    ("training_model",model)
])

In [14]:
# for validating the model , I tried testing for different parameters

full_pipeline.fit(X_train,y_train)

prediction1 = full_pipeline.predict(X_test)

print(accuracy_score(y_test,prediction1))


0.7889591719378953


In [15]:
# Now training model with full training data 

full_pipeline.fit(df,y)

prediction = full_pipeline.predict(df_test)




In [16]:

submission = pd.DataFrame({
    "PassengerId": submission_ids,
    "Transported": prediction 
})
submission.to_csv("submission.csv", index=False)